In [ ]:
import xarray as xr 
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np 
import matplotlib.dates as mdates
import numpy as np 
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd 
import cmocean as cmo
import matplotlib.dates as mdates
import os 
import xarray as xr 
import datetime
from datetime import datetime, timedelta



def julian_to_datetime(julian_days, seconds_, base_julian=2440000, base_date_str="1968-05-23 00:00:00"):
    # 1. Establish your datum anchor point

    julian_days = np.asarray(julian_days)
    seconds_ = np.asarray(seconds_)
    base_date = np.datetime64(base_date_str)
    days_diff = (julian_days - base_julian) * np.timedelta64(1, 'D')
    seconds = seconds_ * np.timedelta64(1, 'ms')
    dates = base_date + days_diff + seconds
    return dates

def format_date_ax(ax, int=1):
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=int))
    date_format = mdates.DateFormatter('%m/%d %H:%M')
    ax.xaxis.set_major_formatter(date_format)
    
ds = xr.open_dataset("hydro.nc", decode_times=False)
print(ds)
nt = len(ds.time) - 1
z = ds.z.values

# ds = ds.isel(time=slice(15, nt))

nt = len(ds.time) - 1

# Load adv data
adv = pd.read_csv("/global/homes/s/siennaw/scratch/siennaw/data/usgs/eps_shear_tke.csv", parse_dates=['time'], index_col='time')
adv.dropna(inplace=True, how='all')
print(adv)


In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt


adv = pd.read_csv("processed_adv_data.csv" , parse_dates=['time'])


# fig = plt.figure(figsize=(12, 6))
# ax = plt.gca() 

# ax.plot(pd.to_datetime(adv['time']), adv['smoothed_shear'], label='Shear', color='k', alpha=0.5)
# ax.plot(pd.to_datetime(adv['time']), adv['shear'], label='Shear', color='blue', alpha=0.2)


adv_shear = adv.copy()
adv_shear.dropna(inplace=True, how='any')
adv_shear = adv_shear[['time', 'shear', 'smoothed_shear']]

adv_shear.index = adv_shear.time 

# adv_shear = adv_shear.iloc[0: 280]
# print(adv_shear)
adv_shear = adv_shear.resample('1s').mean()
adv_shear = adv_shear.interpolate(method='linear', limit_direction='both')
adv_shear.reset_index(drop=True, inplace=True)
adv_shear['seconds'] = np.arange(len(adv_shear))
fig = plt.figure(figsize=(10, 5))
ax = plt.gca()
ax.plot(adv_shear.seconds, adv_shear.shear, color='k', alpha=0.5, label="Original shear")
ax.plot(adv_shear.seconds, adv_shear.smoothed_shear, color='r', label="Smoothed shear")
ax.set_xlabel("Time (s)")
adv_shear.to_csv("adv_shear_4_model.csv")

In [ ]:
# LISST 

# LISST data 
    # Process_Description: Files were named with a convention that uses a 12- to 15-digit alpha-numeric code. 
    #     The first three characters for this dataset are all 'CSF' for the experiment name; the fourth and fifth positions represent the calendar year in which the data were collected (20, 2020); the sixth, seventh and eighth characters are an alphanumeric code for the platform name (CHT, SC1, etc.); the ninth and tenth characters represent the instrument position on the platform, where 01 is the top-most. Next is a two- or three-character code for the instrument type (vec, Nortek's Acoustic Doppler Velocimeter; alt, EofE Ultrasonics' altimeter; wh, RDInstrument's acoustic doppler current profile current data; pt, RBR's bursting pressure sensor with temperature; ctd, RBR's CTD; Tu, RBR's Virtuoso turbidity logger; ls, Sequoia Scientific's LISST; sig, Nortek's Signature1000 ADCP). There are an additional 2-4 characters for instruments that collect in a bursting pattern to indicate whether the file includes the raw burst data (-b), the statistics of the burst (-s), or the wave data (-wvs). This indicator is omitted if there is no bursting pattern.
    #     Process_Date: 20210301

fn = "/global/homes/s/siennaw/scratch/siennaw/data/usgs/CSF20_Shallows_Time_Series/CSF20SC104ls-b.nc"
lds = xr.open_dataset(fn, decode_times=False)
ldates = julian_to_datetime(lds['time'].values, lds['time2'].values)
lds = lds.assign_coords(time=ldates)

print(len(np.unique(ldates))) 
print(len(ldates))
print(lds)

In [ ]:
# Load in sediment data 
from scipy.integrate import cumulative_trapezoid #, trapz
import cmocean as cmo

model = xr.open_dataset("FlocMod_ADV_G_16.nc", decode_times=False)
Ds = model.Ds.values
nf = model.nf
date = pd.to_datetime("2020-07-16 20:10:01")

model = model.dropna(dim="N", how="any")
mtime = [date + pd.Timedelta(t, unit='s') for t in model.time.values]
nt = len(model.time) 



In [ ]:


# # volume concentration
# I = 0 

DsL = lds.size_class.values #x[0:-1]  # LISST size classes

# if 0 :

#     Nl = len(DsL)
#     Nf = len(Ds_)
#     H = np.zeros((Nl, Nf))

#     for j in range(Nf):
#         print("Sorting floc size class: ", Ds_[j])
#         for i in range(Nl):
#             if (Ds_[j] <= DsL[i]):
#                 print("-> placing in LISST size class: ", DsL[i])
#                 H[i, j] = volume[j]
#                 break
#             if i==(Nl-1):
#                 print("Larger than max(LISST) -> placing in LISST size class: ", DsL[i])
#                 H[i, j] = volume[j]
#                 break

#     print(H)

# # Prep for saving everything 

variance = lds.vconc.var(dim='sample').isel(depth=0)


# Theoretical "start date" for our simulation 
start_date = pd.to_datetime("2020-07-16 20:10:01")



time_in_seconds = (lds.time.values - np.datetime64(start_date)) / np.timedelta64(1, 's')

lisst_data = pd.DataFrame()
R = pd.DataFrame()

R['seconds'] = time_in_seconds.astype(int)
lisst_data['seconds'] = time_in_seconds.astype(int)

print(DsL)
# vconc = lds.vconc.mean(dim='sample').isel(depth=0)
for i in range(len(DsL)):
    sed_class = lds.vconc.isel(size_class=i, depth=0) 
    variance = lds.vconc.isel(size_class=i, depth=0).var(dim='sample').values 
    lisst_data[DsL[i]] = sed_class.mean(dim='sample').values
    var_ = variance #variance.isel(size_class=i).values 
    print(np.mean(variance))
    # var_[var_==0] = 1e-2
    R[DsL[i]] = var_

R = R[R['seconds'] >= 0]
lisst_data = lisst_data[lisst_data['seconds'] >= 0]

# Replace zeros in R with mean of that column
for col in R.columns[1:]:
    mean_val = R[col][R[col] != 0].min()  # Calculate mean excluding zeros
    R[col] = R[col].replace(0, mean_val)  # Replace zeros with the mean

R.to_csv("lisst_variance.csv", index=False)
lisst_data.to_csv("lisst_data.csv", index=False)

assert(False)
conc = model.ssc.isel(time=i)
volume = 4/3 * np.pi * (model.Ds/2)**3 
conc = conc*volume 
conc = conc.mean(dim='N').values *1e10

# 4. Plot the CDF against your x-axis (Ds)
color = date2color(mtime[i]) 
interpolated = np.interp(x[0:-1], Ds_, conc) 
axs[1].plot(Ds_, conc, '--', color=color, markersize=3, label=mtime[i].strftime("%b %d %H:%M"))
axs[1].plot(x[0:-1], interpolated, '-o', color=color, markersize=3, label=mtime[i].strftime("%b %d %H:%M"))

# axs[1].plot(Ds_, conc, '-o', color=color, markersize=3, label=mtime[i].strftime("%b %d %H:%M"))

### Plot equivalent LISST data
cdf = lds.vconc.sel(time=mtime[i], method='nearest').isel(depth=0)
cdf = cdf.mean(dim='sample')

d0 = pd.to_datetime(cdf.time.values)
color = date2color(d0) 
axs[0].plot(x[0:-1], cdf, '-o', color=color, linewidth=3, label=d0.strftime("%b %d %H:%M"))

axs[1].set_title("Floc Model Vconc")
axs[0].set_title("LISST Vconc")

for ax in axs:
    ax.set_xlim(1, 1000)
    
    ax.set_xlabel('D$_{50}$ ($\mu$m)')
    # ax.set_xlim(x[0], x[-1])
    ax.grid(alpha=0.2)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

    ax.set_xscale('log')
    ax.set_ylabel("$\mu$L/L")
    print(lds.vconc)

In [ ]:


# lds = lds.isel(depth=0)



# stds = lds['D50'].std(axis=1) 
# d50 = lds['D50'].mean(axis=1)


# # assert(False)

# for d in range(17, 28):

#     fig = plt.figure(figsize=(10,4))
#     ax = plt.gca()

#     ax.fill_between(ldates, np.squeeze(d50-stds), np.squeeze(d50+stds), color='#CBE2FE', alpha=0.55, label="Standard Deviation")

#     ax.plot(ldates, d50, 'o', color='#10288C', label="D$_{50}$")
#     # fill between

#     ax.grid(alpha=0.25)
#     ax.set_ylabel('D$_{50}$ ($\mu$m)')
#     ax.set_ylim(0,)
#     ax.set_title('D$_{50}$ from LISST')
#     format_date_ax(ax, 12)
#     ax.set_xlim(pd.to_datetime('2020-07-%d' % d), pd.to_datetime('2020-07-%d' % (d+1)))
#     ax.legend()

#### 


In [ ]:
# Load in ADCP data
adcp = xr.open_dataset("/global/homes/s/siennaw/scratch/siennaw/data/usgs/resampled_adcp_usgs.nc") #, decode_times=False)

rtime = adcp.time.values
U = adcp.u_1205.mean(dim='depth').values
V = adcp.v_1206.mean(dim='depth').values


lateral_velocity = np.squeeze(np.sqrt(U**2 + V**2)) * 0.01
print(lateral_velocity)

fig = plt.figure(figsize=(12, 4))
ax = plt.gca()
ax.plot(rtime, lateral_velocity, color='k', alpha=0.8)
ax.set_ylabel("Horizontal flow (m/s)")
ax.grid(alpha=0.2)
ax.set_xlim(rtime[0], rtime[-1])
format_date_ax(ax, 48)
fig.savefig("adcp_lateral_velocity.png", dpi=300, bbox_inches='tight')


# Prep for saving 

# Theoretical "start date" for our simulation 
start_date = pd.to_datetime("2020-07-16 20:10:01")

time_in_seconds = (rtime - np.datetime64(start_date)) / np.timedelta64(1, 's')

adcp_data = pd.DataFrame()
adcp_data['seconds'] = time_in_seconds.astype(int)
adcp_data['lateral_velocity'] = lateral_velocity
adcp_data['u'] = np.squeeze(U)*0.01
adcp_data['v'] = np.squeeze(V)*0.01
adcp_data.to_csv("adcp_lateral_velocity.csv", index=False)


In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

low, high = 0, 400
rtime = adcp.time.values[low:high]

offset= 3.537306921274909


axs[0].set_xlim(rtime[0], rtime[-1])
levels = [-100, -80, -50, -20, -1, 0, 1, 20, 50, 80, 100]

velocity = np.squeeze(adcp.u_1205)
z = adcp.depth.values
h = axs[0].contourf(rtime, -z + offset, velocity[low:high,:].T, cmap=cmo.cm.balance, levels=levels)
plt.colorbar(h, label='U Velocity (cm/s)', shrink=0.7)
axs[0].set_title('Eastward Velocity (U)')

def format_date_ax(ax, int=1):
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=int))
    date_format = mdates.DateFormatter('%m/%d %H:%M')
    ax.xaxis.set_major_formatter(date_format)


depths =  np.squeeze(adcp.P_1.values[low:high]) #+ 0.41

axs[0].plot(rtime, depths, color='k', label='Depth')
axs[1].plot(rtime, depths, color='k', label='Depth')
axs[2].plot(rtime, depths, color='k', label='Depth')

velocity = np.squeeze(adcp.v_1206)
h = axs[1].contourf(rtime, -z + offset, velocity[low:high,:].T,  cmap=cmo.cm.balance, levels=levels)

plt.colorbar(h, label='V Velocity (cm/s)', shrink=0.7)
axs[1].set_title('Northward Velocity (V)')

vmin, vmax = -10, 10
velocity = np.squeeze(adcp.w_1204)

levels = [-9, -4, -1, 0, 1, 4, 9]
h = axs[2].contourf(rtime, -z + offset, velocity[low:high,:].T, cmap=cmo.cm.balance, levels=levels)
plt.colorbar(h, label='W Velocity (cm/s)', shrink=0.7)
axs[2].set_title('Upward Velocity (W)')

for ax in axs:
    ax.grid(alpha=0.25)
    format_date_ax(ax, int=6)
    ax.set_ylim(0, 5.5)


In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(10, 7), sharex=False, constrained_layout=True)

low, high = 0, 900
rtime = adcp.time.values[low:high]

#                                 # blanking distance         
depths =  np.squeeze(adcp.P_1.values[low:high])  

levels = [-100, -80, -50, -20, -1, 0, 1, 20, 50, 80, 100]

velocity = np.squeeze(adcp.u_1205)
velocity = velocity[low:high,:].T
z = adcp.depth.values
h = axs[1].contourf(rtime, offset-z, velocity, cmap=cmo.cm.balance, levels=levels)
plt.colorbar(h, label='U Velocity (cm/s)', shrink=0.7)
axs[1].set_title('ADCP (u, eastward)')


velocity = np.squeeze(adcp.w_1204)
levelsW = [-4, -1, -0.5, 0, 0.5, 1, 4]
h = axs[0].contourf(rtime, offset-z, velocity[low:high,:].T, cmap=cmo.cm.balance, levels=levelsW)
plt.colorbar(h, label='w (cm/s)', shrink=0.7)
axs[0].set_title('ADCP (w, upward)')


model_time = [pd.to_datetime(rtime[0]) + pd.Timedelta(seconds=j) for j in ds.time.values]

axs[2].set_title("Modeled U Velocity (eastward)")
h = axs[2].contourf(model_time, 4-ds.z, ds.U.T*100, cmap=cmo.cm.balance, levels=levels)
plt.colorbar(h, label='U Velocity (cm/s)', shrink=0.7)

axs[3].set_title("Modeled shear")
shear = np.sqrt((ds.Q2 /ds.Kz)) 
ax2 = axs[3].twinx()
ax2.plot(model_time, shear.sel(z=0.1, method='nearest').values, '--', color='b', label='modeled shear @ 0.2 m')
ax2.set_ylim(0, 2)

adv_shear = adv.shear.dropna()
ax2.plot(adv_shear/10, color='r', label='ADV Shear')
ax2.legend()
h = axs[3].pcolormesh(model_time, 4-ds.z, shear.T, cmap=cmo.cm.rain, vmin=0, vmax=2)
ax2.set_ylabel('shear (1/s)')
# h = axs[3].pcolormesh(model_time, 4-ds.z, ds.C.T, cmap=cmo.cm.rain, vmin=20, vmax=25)

# h = axs[3].pcolormesh(model_time, ds.z, np.log(ds.Q2.T), cmap=cmo.cm.rain, vmin=-8, vmax=-2)
plt.colorbar(h, label='shear (1/s)', shrink=0.7)

for ax in axs:
    ax.plot(rtime, depths, color='k', label='Depth')
    ax.grid(alpha=0.25)
    format_date_ax(ax, int=12)
    ax.set_xlim(rtime[0], rtime[-1])
    ax.set_ylim(0, 10)
    ax.set_ylabel('Depth (m)')
# plt.tight_layout()

